# Lecture 14 – Visualizations III

### DATA 2201, Fall 2025



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Lecture Outline:
- Transformations  
- Theory of Visualization with Demo  
  - Current Population Survey  
  - CO2 Emissions

### World Bank Dataset

In [ ]:
wb = pd.read_csv("data/world_bank.csv", index_col=0)
wb = wb.rename(columns={'Antiretroviral therapy coverage: % of people living with HIV: 2015':"HIV rate",
                       'Gross national income per capita, Atlas method: $: 2016':'gni'})
wb.head()

# Transformations
When we visualize relationships between variables, our goal is often to understand how one variable changes with another so that we can later model this relationship. In **DATA 2201**, we’ll spend much of the semester working with **linear models**, which describe relationships using straight lines.  

To make data more suitable for linear modeling, we sometimes apply **transformations**—mathematical adjustments such as taking the log, square root, or reciprocal of a variable. These transformations help us in several ways:

- **Clarify patterns:** They make trends in data easier to see and interpret visually.  
- **Linearize relationships:** Many relationships that appear curved or nonlinear become approximately linear after transformation.  
- **Simplify interpretation:** Linear relationships are intuitive—slopes and intercepts have clear, real-world meanings.  
- **Improve modeling:** Later in the course, being able to linearize data will help us build more accurate and reliable models.

In short, transformations are a bridge between visualization and modeling, helping us see structure in data that supports meaningful analysis.



In [ ]:
# Some data cleaning to help with the next example

df = pd.DataFrame(index=wb.index)
df['lit'] = wb['Adult literacy rate: Female: % ages 15 and older: 2005-14'] \
            + wb["Adult literacy rate: Male: % ages 15 and older: 2005-14"]
df['inc'] = wb['gni']
df.dropna(inplace=True)

plt.scatter(df["inc"], df["lit"])
plt.xlabel("Gross national income per capita")
plt.ylabel("Adult literacy rate");

### Why the Plot Appears Non-Linear
The relationship between **gross national income (GNI)** and **adult literacy rate** looks curved rather than linear because of how the data are distributed.

- A few countries have **very large GNI values**, which stretch the x-axis and distort the overall scale.  
- Most data points with smaller GNI values get **compressed on the left side**, hiding the true pattern.  
- The clustering of high literacy rates at the top makes it harder to see variation across countries.  

To fix this, we can apply a **log transformation** to the GNI values:
- The log transformation **compresses large numbers** and **spreads out smaller ones**, making the scale more balanced.  
- This rescaling reveals a clearer, **more linear relationship** between income and literacy rate.


In [ ]:
# np.log compute the natural (base e) logarithm
plt.scatter(np.log(df["inc"]), df["lit"])
plt.xlabel("Log(gross national income per capita)")
plt.ylabel("Adult literacy rate");

Already, the relationship is starting to look more linear! Now, we'll address the vertical scaling. 

- To reduce the clumping of datapoints near the top of the plot, we want to spread out large values of y without substantially changing small values of y.
- We can do this by applying a **power transformation** – that is, by raising the y-values to a power. Below, we raise all y-values to the power of 4.

In [ ]:
plt.scatter(np.log(df["inc"]), df["lit"]**4)
plt.xlabel("Log(gross national income per capita)")
plt.ylabel("Adult literacy rate (4th power)");

Our transformed variables now seem to follow a linear relationship! 

$$y^4 = m(\log{x}) + b$$

We can use this fact to uncover new information about the original, untransformed variables. 

$$y = [m(\log{x}) + b]^{1/4}$$

#### In the cell below: 
- We first fit a regression line to the transformed data to find values for the slope ($m$) and intercept ($b$).
- Then, we plug these values into the relationship we derived for the *untransformed* variables.
- We find a mathematical relationship relating the gross national income and the adult literacy rate.

In [ ]:
# The code below fits a linear regression model. We'll discuss it at length in a future lecture
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(np.log(df[["inc"]]), df["lit"]**4)
m, b = model.coef_[0], model.intercept_

print(f"The slope, m, of the transformed data is: {m}")
print(f"The intercept, b, of the transformed data is: {b}")

df = df.sort_values("inc")
plt.scatter(np.log(df["inc"]), df["lit"]**4, label="Transformed data")
plt.plot(np.log(df["inc"]), m*np.log(df["inc"])+b, c="red", label="Linear regression")
plt.xlabel("Log(gross national income per capita)")
plt.ylabel("Adult literacy rate (4th power)")
plt.legend();

In [ ]:
# Now, plug the values for m and b into the relationship between the untransformed x and y
plt.scatter(df["inc"], df["lit"], label="Untransformed data")
plt.plot(df["inc"], (m*np.log(df["inc"])+b)**(1/4), c="red", label="Modeled relationship")
plt.xlabel("Gross national income per capita")
plt.ylabel("Adult literacy rate")
plt.legend();

We've been able to find a fairly close approximation for the relationship between the original variables!

# Theory of Visualization
## Scale Demo:
### Summary of the Demo  
- Shows how to visualize changes in Planned Parenthood service counts (Cancer, Abortion) between 2006 and 2013.  
- Uses a small DataFrame to store counts for the two years.  
- Line plot illustrates overall trends in service counts.  
- Calculates percent change between years for each service.  
- Bar chart visualizes which services increased or decreased over time.


In [ ]:
ppdf = pd.DataFrame(dict(Cancer=[2007371, 935573], 
                         Abortion=[289750, 327000]), 
                    index=pd.Series([2006, 2013], 
                    name="Year"))
ppdf

In [ ]:
ax = sns.lineplot(data=ppdf, markers=True)
ax.set_title("Planned Parenthood Procedures")
ax.set_xticks([2006, 2013])
ax.set_ylabel("Service count");

Let's now compute the relative change between the two years...

In [ ]:
rel_change = 100*(ppdf.loc[2013] - ppdf.loc[2006])/ppdf.loc[2006]
rel_change.name = "Percent Change"
rel_change

In [ ]:
ax = sns.barplot(x=rel_change.index, y=rel_change)
ax.axhline(0, color='black')
ax.set_title("Percent Change in Number of Procedures");

## Current Population Survey
### Summary of the Demo – Current Population Survey  
- Demonstrates how to visualize income differences by education level and gender using U.S. Current Population Survey data.  
- Reads and cleans the dataset, replacing numeric education codes with readable labels.  
- Uses **Seaborn’s `pointplot()`** with a custom blue–red color palette to compare male and female median weekly earnings across education levels.  
- The plot highlights the **income gap** and how it changes with higher education attainment.


In [ ]:
cps = pd.read_csv("data/edInc2.csv")
cps

In [ ]:
cps = cps.replace({'educ':{1:"<HS", 2:"HS", 3:"<BA", 4:"BA", 5:">BA"}})
cps.columns = ['Education', 'Gender', 'Income']
cps

In [ ]:
# Let's pick our colors specifically using color_palette()
blue_red = ["#397eb7", "#bf1518"]
with sns.color_palette(sns.color_palette(blue_red)):
    ax = sns.pointplot(data=cps, x="Education", y="Income", hue="Gender")

ax.set_title("2014 Median Weekly Earnings\nFull-Time Workers over 25 years old");

Now, let's compute the income gap as a relative quantity between men and women. Recall that the structure of the dataframe is as follows:

In [ ]:
cps.head()

## This calls for using `groupby` by Gender, so that we can separate the data for both genders, and then compute the ratio.

#### Groups the dataset by **Gender** and extracts separate data for men and women.  

In [ ]:
cg = cps.set_index("Education").groupby("Gender")
men = cg.get_group("Men").drop("Gender", axis="columns")
women = cg.get_group("Women").drop("Gender", axis="columns")
display(men, women)

#### Computes the **Male-to-Female (M/F) income ratio** for each education level to show how men’s earnings compare to women’s. 

In [ ]:
mfratio = men/women
mfratio.columns = ["Income Ratio (M/F)"]
mfratio

#### Plots the ratio using a line chart labeled “M/F Income Ratio as a Function of Education Level.”  

In [ ]:
ax = sns.lineplot(data=mfratio, markers=True, legend=False);
ax.set_ylabel("Ratio")
ax.set_title("M/F Income Ratio as a function of education level");

Let's now compute the alternate ratio, F/M instead:

In [ ]:
fmratio = women/men
fmratio.columns = ["Income Ratio (F/M)"]
fmratio

#### Visualizes both ratios to highlight the **gender income gap** across education levels.

In [ ]:
ax = sns.lineplot(data=fmratio, markers=True, legend=False);
ax.set_ylabel("Ratio")
ax.set_title("F/M Income Ratio as a function of education level");

## CO2 Emissions 

### Summary of the Demo – CO₂ Emissions  
- Reads a dataset of **country-level CO₂ emissions** and filters it to focus on the **top 14 emitting countries** in the most recent year available.  
- Cleans and subsets the data to include emissions from **1950 onward** for consistent comparison.  
- Uses **custom colors and line styles** (via the `cycler` module) for clear visualization of multiple countries.  
- Plots CO₂ emissions over time on a **logarithmic (semilogy) scale**, making exponential growth patterns easier to compare.  
- The visualization highlights global emission trends and differences among major contributors to climate change.


In [ ]:
co2 = pd.read_csv("data/CAITcountryCO2.csv", skiprows=2,
                  names=["Country", "Year", "CO2"], encoding="ISO-8859-1")
co2.tail()

In [ ]:
last_year = co2.Year.iloc[-1]
last_year

In [ ]:
q = f"Country != 'World' and Country != 'European Union (15)' and Year == {last_year}"
top14_lasty = co2.query(q).sort_values('CO2', ascending=False).iloc[:14]
top14_lasty

In [ ]:
top14 = co2[co2.Country.isin(top14_lasty.Country) & (co2.Year >= 1950)]
print(len(top14.Country.unique()))
top14.head()

In [ ]:
from cycler import cycler

linestyles = ['-', '--', ':', '-.' ]
colors = plt.cm.Dark2.colors
lines_c = cycler('linestyle', linestyles)
color_c = cycler('color', colors)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_prop_cycle(color_c * lines_c)

x, y ='Year', 'CO2'
for name, df in top14.groupby('Country'):
    ax.semilogy(df[x], df[y], label=name)

ax.set_xlabel(x)
ax.set_ylabel(f"{y} Emissions (million tons)")
ax.legend(ncol=2, frameon=True, fontsize=11);

# See you next lecture!